In [42]:
def reverse_complement_64(kmer_bin: int, kmer_size: int) -> int:
    """Compute the reverse complement of a 2-bit encoded k-mer packed in an int.

    Assumes kmer_bin uses 2*kmer_size bits. Mimics the bitwise operations in the C++ ReverseComp64 function.
    """
    # Step 1: Complement the k-mer (note: only flip the lower 2*k bits)
    mask = (1 << (2 * kmer_size)) - 1
    res = ~kmer_bin & mask

    # Step 2: Reverse the bits in 2-bit chunks (bit twiddling from C++)
    res = ((res >> 2) & 0x3333333333333333) | ((res & 0x3333333333333333) << 2)
    res = ((res >> 4) & 0x0F0F0F0F0F0F0F0F) | ((res & 0x0F0F0F0F0F0F0F0F) << 4)
    res = ((res >> 8) & 0x00FF00FF00FF00FF) | ((res & 0x00FF00FF00FF00FF) << 8)
    res = ((res >> 16) & 0x0000FFFF0000FFFF) | ((res & 0x0000FFFF0000FFFF) << 16)
    res = ((res >> 32) & 0x00000000FFFFFFFF) | ((res & 0x00000000FFFFFFFF) << 32)

    # Step 3: Shift to align the result to the rightmost bits
    res >>= 2 * (32 - kmer_size)

    return res

def kmer_to_binary(kmer: str) -> int:
    base_to_bits = {'A': 0b00, 'C': 0b01, 'G': 0b10, 'T': 0b11}
    binary = 0
    for base in kmer:
        binary = (binary << 2) | base_to_bits[base]
    return binary

def canonical_kmer_binary(kmer: str) -> tuple[int, bool]:
    kmer_bin = kmer_to_binary(kmer)
    revcomp_bin = reverse_complement_64(kmer_bin, len(kmer))
    return (min(kmer_bin, revcomp_bin), revcomp_bin < kmer_bin)

def extract_bits(kmer_bin, nuc_x, k, rc):
    mask = 0b11 << (k-nuc_x-1) * 2
    extracted_bits = (kmer_bin & mask) >> (k-nuc_x-1) * 2
    if rc:
        extracted_bits = (~extracted_bits) & ((1 << 2) - 1)
    return extracted_bits

def extract_bits2(kmer_bin: int, nuc_x: int, k: int, rc: bool) -> int:
    """Extract the original nucleotide at position nuc_x from a k-mer binary.

    kmer_bin: binary representation (forward or reverse complement)
    nuc_x: index (0-based) in original forward k-mer
    k: length of the k-mer
    rc: whether kmer_bin is reverse complement
    Returns: 2-bit integer (00=A, 01=C, 10=G, 11=T)
    """
    if rc:
        # If it's a reverse complement, the original nuc_x corresponds to k - nuc_x - 1
        pos = k - nuc_x - 1
    else:
        pos = nuc_x

    extracted_bits = (kmer_bin >> (2 * (k - pos - 1))) & 0b11

    if rc:
        # Convert back from reverse complement
        extracted_bits ^= 0b11

    return extracted_bits


In [45]:
k_bin, rev = canonical_kmer_binary("GCC")
print(bin(k_bin), k_bin, rev)

0b100101 37 False


In [46]:
extract_bits2(k_bin, 0, 3, rev)

2